## Creating a Single Table from Multiple Snapshots

It is common to receive data as **periodic snapshots** — e.g. a termly extract of pupil records, a monthly export from a finance system, or a daily feed from an API. Each snapshot has the same structure but covers a different point in time.

To analyse trends or build a complete picture, you need to combine these into a single table.

### Key principles

* **Use `UNION ALL`**, not `UNION`. `UNION` silently removes duplicates, which hides data quality issues and is slower. Always prefer `UNION ALL` and handle duplicates explicitly.
* **Add a snapshot identifier.** Tag each extract so you always know which source a row came from. This is essential for debugging and for time-series analysis.
* **Validate column alignment.** Snapshots may drift over time — columns renamed, added, or removed. Check schemas match before combining.

### Watch out for

* **Overlapping snapshots** — if two extracts cover the same period, you’ll get genuine duplicates
* **Schema drift** — a column called `dob` in one snapshot and `date_of_birth` in another
* **Changed grain** — one snapshot at pupil level, another at pupil-school level

In [0]:
%sql
-- Two termly snapshots of pupil data with the same structure

CREATE OR REPLACE TEMP VIEW autumn_snapshot AS
SELECT * FROM VALUES
  (1, 'Alice', 100, 'Oak Academy',  'Year 7'),
  (2, 'Bob',   100, 'Oak Academy',  'Year 8'),
  (3, 'Carol', 200, 'Elm School',   'Year 7')
AS t(pupil_id, pupil_name, school_urn, school_name, year_group);

CREATE OR REPLACE TEMP VIEW spring_snapshot AS
SELECT * FROM VALUES
  (1, 'Alice', 200, 'Elm School',   'Year 7'),   -- Alice moved school
  (2, 'Bob',   100, 'Oak Academy',  'Year 8'),
  (3, 'Carol', 200, 'Elm School',   'Year 7'),
  (4, 'Dan',   300, 'Birch College','Year 9')    -- New pupil
AS t(pupil_id, pupil_name, school_urn, school_name, year_group);

In [0]:
%sql
-- Combine snapshots with a snapshot_term column to track origin
CREATE OR REPLACE TEMP VIEW all_terms AS
SELECT *, 'Autumn' AS snapshot_term FROM autumn_snapshot
UNION ALL
SELECT *, 'Spring' AS snapshot_term FROM spring_snapshot;

SELECT * FROM all_terms
ORDER BY pupil_id, snapshot_term;

In [0]:
%sql
-- Quick validation: row counts should equal the sum of individual snapshots
SELECT
  (SELECT COUNT(*) FROM autumn_snapshot)  AS autumn_rows,
  (SELECT COUNT(*) FROM spring_snapshot)  AS spring_rows,
  (SELECT COUNT(*) FROM all_terms)        AS combined_rows,
  CASE
    WHEN (SELECT COUNT(*) FROM autumn_snapshot) + (SELECT COUNT(*) FROM spring_snapshot)
       = (SELECT COUNT(*) FROM all_terms)
    THEN 'Row counts match - UNION ALL is correct'
    ELSE 'Row count mismatch - investigate'
  END AS validation;